# Data Scraping and Cleaning
> **Note** this notebook for data scraping and cleaning was used with assistance from Claude. 

In [1]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time
import logging
from pathlib import Path

## Steam Charts

### Set up

In [2]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/122.0.0.0 Safari/537.36"
}

STEAMCHARTS_URL = "https://steamcharts.com/app/{app_id}#All"

RAW_DIR = Path("data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Pause between requests
DELAY_SECONDS = 3 

In [5]:
df1 = pd.read_csv("data/raw/top50_free_games_list.csv")
df2 = pd.read_csv("data/raw/top50_paid_games_list.csv")

display(df1.head())
display(df2.head())

,AppID,Game Title,Release Date,Estimated Owners,Peak Concurrent Players (24h)
0,730,Counter-Strike 2,2012-08-21,100M+,1415913
1,570,Dota 2,2013-07-09,100M+,632090
2,578080,PUBG: BATTLEGROUNDS,2017-12-21,50M - 100M,798913
3,1172470,Apex Legends,2020-11-04,50M - 100M,277066
4,230410,Warframe,2013-03-25,50M - 100M,75491


,AppID,Game Title,Release Date,Base Price,Estimated Owners,Peak Concurrent Players (24h)
0,271590,Grand Theft Auto V Legacy,2015-04-13,$29.99,50M - 100M,88110
1,105600,Terraria,2011-05-16,$9.99,20M - 50M,37858
2,292030,The Witcher 3: Wild Hunt,2015-05-18,$39.99,20M - 50M,17925
3,4000,Garry's Mod,2006-11-29,$9.99,20M - 50M,30201
4,550,Left 4 Dead 2,2009-11-17,$9.99,20M - 50M,18706


In [11]:
def scrape_steamcharts(app_id: int, game_name: str) -> list[dict] | None:
    url = f"https://steamcharts.com/app/{app_id}"

    try:
        response = requests.get(url, headers=HEADERS, timeout=10)

        if response.status_code == 404:
            logging.warning(f"[{app_id}] '{game_name}' not found on SteamCharts (404). Skipping.")
            return None

        response.raise_for_status()

    except requests.RequestException as e:
        logging.warning(f"[{app_id}] '{game_name}' request failed: {e}. Skipping.")
        return None

    soup = BeautifulSoup(response.text, "html.parser")
    table = soup.find("table", {"class": "common-table"})  # fixed selector

    if not table:
        logging.warning(f"[{app_id}] '{game_name}' — table not found on page. Skipping.")
        return None

    rows = table.find("tbody").find_all("tr")
    records = []

    for row in rows:
        cols = [td.get_text(strip=True) for td in row.find_all("td")]
        if len(cols) < 3:
            continue

        year_month_raw  = cols[0]
        avg_players_raw = cols[1]
        peak_players_raw = cols[4]  # col index 4 based on debug output

        # Skip the rolling "Last 30 Days" row
        if year_month_raw == "Last 30 Days":
            continue

        try:
            year_month = pd.to_datetime(year_month_raw, format="%B %Y").strftime("%Y-%m")
            avg  = float(avg_players_raw.replace(",", ""))
            peak = int(peak_players_raw.replace(",", ""))
        except ValueError:
            logging.warning(f"[{app_id}] Could not parse row: {cols}. Skipping row.")
            continue

        records.append({
            "game_id":              app_id,
            "game_name":            game_name,
            "year_month":           year_month,
            "peak_player_count":    peak,
            "average_player_count": avg,
        })

    return records if records else None

In [7]:
def scrape_dataframe(df: pd.DataFrame, label: str) -> pd.DataFrame:
    """
    Iterates over a games DataFrame, scrapes SteamCharts for each,
    and returns a combined DataFrame.
    """
    all_records = []
    total = len(df)

    for i, row in enumerate(df.itertuples(), start=1):
        app_id = int(row.AppID)
        game_name = row._2  # "Game Title" — itertuples sanitizes column names with spaces

        logging.info(f"[{label}] ({i}/{total}) Scraping '{game_name}' (ID: {app_id})...")
        records = scrape_steamcharts(app_id, game_name)

        if records:
            all_records.extend(records)

        if i < total:
            time.sleep(DELAY_SECONDS)

    return pd.DataFrame(all_records)

### Scraping

This is a guard variable to prevent the next cells from being 
executed as these scrape data from the internet. To prevent 
the abuse of internet, this is a safeguard. Make this variable
True if the data needs to be scraped again.

In [23]:
WANT_TO_SCRAPE = False

In [ ]:
if not WANT_TO_SCRAPE:
    raise RuntimeError("See the first cell under \'Scraping\'.")

results_1 = scrape_dataframe(df1, label="CSV-1")
print(f"Total rows scraped: {len(df1)}")

In [ ]:
if not WANT_TO_SCRAPE:
    raise RuntimeError("See the first cell under \'Scraping\'.")

results_2 = scrape_dataframe(df2, label="CSV-2")
print(f"Total rows scraped: {len(df2)}")

In [ ]:
if not WANT_TO_SCRAPE:
    raise RuntimeError("See the first cell under \'Scraping\'.")

out_1 = RAW_DIR / "top50_free_games_player_data.csv"
out_2 = RAW_DIR / "top50_paid_games_player_data.csv"

results_1.to_csv(out_1, index=False)
results_2.to_csv(out_2, index=False)

logging.info(f"Saved: {out_1}")
logging.info(f"Saved: {out_2}")

## Sanity Checks

In [24]:
EXPECTED_GAMES = 50
EXPECTED_GAMES_MIN = 45

for label, df, source_csv in [
    ("CSV-1", results_1, df1),
    ("CSV-2", results_2, df2),
]:
    print(f"{'='*40}")
    print(f" {label}")
    print(f"{'='*40}")

    scraped_ids = set(df["game_id"].unique())
    all_ids     = set(source_csv["AppID"].astype(int).unique())
    skipped     = all_ids - scraped_ids

    # 1. Game count
    n_games = df["game_id"].nunique()
    game_ok = EXPECTED_GAMES_MIN <= n_games <= EXPECTED_GAMES
    print(f"[{'OK' if game_ok else '!!'}] Unique games scraped : {n_games} (expected {EXPECTED_GAMES_MIN}–{EXPECTED_GAMES})")

    # 2. Skipped games
    print(f"[{'OK' if len(skipped) == 0 else '--'}] Skipped games        : {len(skipped)}")
    for app_id in skipped:
        name = source_csv.loc[source_csv["AppID"] == app_id, "Game Title"].values[0]
        print(f"      {app_id} — {name}")

    # 3. Row count sanity (at least 12 months per game on average)
    avg_rows_per_game = len(df) / n_games if n_games > 0 else 0
    rows_ok = avg_rows_per_game >= 12
    print(f"[{'OK' if rows_ok else '!!'}] Avg rows per game    : {avg_rows_per_game:.1f} (expected ≥ 12)")

    # 4. No missing values in key columns
    nulls = df[["game_id", "game_name", "year_month", "peak_player_count", "average_player_count"]].isnull().sum()
    nulls_ok = nulls.sum() == 0
    print(f"[{'OK' if nulls_ok else '!!'}] Null values          : {nulls.sum()}")
    if not nulls_ok:
        print(nulls[nulls > 0])

    # 5. year_month format looks right
    bad_dates = df[~df["year_month"].str.match(r"^\d{4}-\d{2}$")]
    dates_ok = len(bad_dates) == 0
    print(f"[{'OK' if dates_ok else '!!'}] Malformed dates      : {len(bad_dates)}")

    # 6. No duplicate (game_id, year_month) pairs
    dupes = df.duplicated(subset=["game_id", "year_month"]).sum()
    dupes_ok = dupes == 0
    print(f"[{'OK' if dupes_ok else '!!'}] Duplicate rows       : {dupes}")

    # 7. Player counts are non-negative
    neg_peak = (df["peak_player_count"] < 0).sum()
    neg_avg  = (df["average_player_count"] < 0).sum()
    counts_ok = neg_peak == 0 and neg_avg == 0
    print(f"[{'OK' if counts_ok else '!!'}] Negative player counts: peak={neg_peak}, avg={neg_avg}")

    print()

 CSV-1
[OK] Unique games scraped : 47 (expected 45–50)
[--] Skipped games        : 3
      409160 — Dr. Langeskov The Tiger and The Emerald
      365300 — Transmissions: Element 120
      1121910 — I Love You Colonel Sanders!
[OK] Avg rows per game    : 106.7 (expected ≥ 12)
[OK] Null values          : 0
[OK] Malformed dates      : 0
[OK] Duplicate rows       : 0
[OK] Negative player counts: peak=0, avg=0

 CSV-2
[OK] Unique games scraped : 50 (expected 45–50)
[OK] Skipped games        : 0
[OK] Avg rows per game    : 104.2 (expected ≥ 12)
[OK] Null values          : 0
[OK] Malformed dates      : 0
[OK] Duplicate rows       : 0
[OK] Negative player counts: peak=0, avg=0



## Steam Reviews
> **Note** this section for scraping Steam review scores was written with assistance from Claude.

For each game in the two lists loaded above, this calls Steam's `appreviews` endpoint and records the current all-time review summary: percent positive, the rating label, and the positive / negative / total review counts.

### Set up

In [ ]:
APPREVIEWS_URL = (
    "https://store.steampowered.com/appreviews/{app_id}"
    "?json=1&language=all&purchase_type=all&num_per_page=0"
)

# Review-score tables live alongside the other cleaned data files.
DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def scrape_review_summary(app_id: int, game_name: str) -> dict | None:
    url = APPREVIEWS_URL.format(app_id=app_id)

    try:
        response = requests.get(url, headers=HEADERS, timeout=20)
        response.raise_for_status()
        payload = response.json()

    except requests.RequestException as e:
        logging.warning(f"[{app_id}] '{game_name}' request failed: {e}. Skipping.")
        return None
    except ValueError as e:
        logging.warning(f"[{app_id}] '{game_name}' returned invalid JSON: {e}. Skipping.")
        return None

    if payload.get("success") != 1:
        logging.warning(f"[{app_id}] '{game_name}' — appreviews returned no summary. Skipping.")
        return None

    summary = payload.get("query_summary", {})
    positive = int(summary.get("total_positive", 0))
    negative = int(summary.get("total_negative", 0))
    total = int(summary.get("total_reviews", positive + negative))
    score = round(100 * positive / total) if total else None

    return {
        "AppID":               app_id,
        "Game Title":          game_name,
        "Review Score (/100)": score,
        "Rating":              summary.get("review_score_desc", ""),
        "Total Reviews":       total,
        "Positive":            positive,
        "Negative":            negative,
    }

In [ ]:
def scrape_reviews_dataframe(df: pd.DataFrame, label: str) -> pd.DataFrame:
    """
    Iterates over a games DataFrame, fetches the all-time Steam review
    summary for each game, and returns a combined DataFrame.
    """
    all_records = []
    total = len(df)

    for i, row in enumerate(df.itertuples(), start=1):
        app_id = int(row.AppID)
        game_name = row._2  # "Game Title" — itertuples sanitizes column names with spaces

        logging.info(f"[{label}] ({i}/{total}) Fetching reviews for '{game_name}' (ID: {app_id})...")
        record = scrape_review_summary(app_id, game_name)

        if record:
            all_records.append(record)

        if i < total:
            time.sleep(DELAY_SECONDS)

    return pd.DataFrame(all_records)

### Scraping
These cells hit the network, so they reuse the same `WANT_TO_SCRAPE` guard defined under "## Steam Charts" → "### Scraping". Set `WANT_TO_SCRAPE = True` there to re-fetch the review scores.

In [ ]:
if not WANT_TO_SCRAPE:
    raise RuntimeError("See the first cell under 'Scraping'.")

reviews_1 = scrape_reviews_dataframe(df1, label="CSV-1")
print(f"Total games fetched: {reviews_1['AppID'].nunique()}")

In [ ]:
if not WANT_TO_SCRAPE:
    raise RuntimeError("See the first cell under 'Scraping'.")

reviews_2 = scrape_reviews_dataframe(df2, label="CSV-2")
print(f"Total games fetched: {reviews_2['AppID'].nunique()}")

In [ ]:
if not WANT_TO_SCRAPE:
    raise RuntimeError("See the first cell under 'Scraping'.")

reviews_out_1 = DATA_DIR / "top50_free_games_review_scores.csv"
reviews_out_2 = DATA_DIR / "top50_paid_games_review_scores.csv"

reviews_1.to_csv(reviews_out_1, index=False)
reviews_2.to_csv(reviews_out_2, index=False)

logging.info(f"Saved: {reviews_out_1}")
logging.info(f"Saved: {reviews_out_2}")